# CADAC ROCKET6G — gray-box training

Trains the learned correction on a dataset produced by
[`colab_generate_data.ipynb`](colab_generate_data.ipynb).

**Pipeline:** load → **residual gate** → train → inspect `dA` → roll out → identifiability.

### What is actually being identified

The state is 6 wide (`SBII1..3`, `VBII1..3`) so `A` is 6×6. Of its 36 entries,
**18 are frozen**: the kinematic rows `d(SBII)/dt = VBII` are a definition, not a
model, and `free_mask` zeroes them so the network cannot spend capacity
contradicting an identity. The same applies to the first three entries of `c`.

That leaves **18 `dA` entries + 3 `dc` entries = 21 unknowns**, re-emitted at every
sample, against **3 equations** (the acceleration residual on the `VBII` rows). The
system is wildly underdetermined, and `lambda_reg` is the only thing making the
answer unique. Section 9 checks whether that actually worked.

The deliverable is not the network. It is `dA[3:6, 3:6]`, which has units of 1/s and
multiplies velocity — that block **is** the identified drag model.

### Read the gate before you read the loss

Section 4 is not a formality. With aerodynamics the only unknown, the residual left
by analytical physics must be near zero in near-vacuum and **grow with `pdynmc`**.
A residual that is flat across dynamic-pressure buckets means a physics module is
wrong, and training would bury that error inside `dA` labelled as aerodynamics —
producing a good loss curve and a meaningless answer.

## 1. Environment

In [ ]:
import sys, subprocess, pathlib, time
import torch

print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU runtime)')

A GPU helps the training loop (batch 2048 through a 24.9k-parameter MLP) but does
**nothing** for section 10: `evaluate._euler` is a batch-size-1 Python loop, so the
rollout is bound by per-call overhead, not arithmetic. A CPU runtime is a perfectly
reasonable choice here.

import os, sys, shutil, subprocess, pathlib

REPO = 'https://github.com/alican30alicanexe-alt/systemid'
CODE = pathlib.Path('/content/systemid')

# Always refresh; never trust an existing checkout.
#
# This cell used to skip the clone whenever CODE/*.py already existed. A runtime
# that had cloned earlier -- before a push, or in a previous session -- therefore
# kept running old code while the notebook itself was new. That produced a full
# 50-run dataset against the pre-FSPB schema before anything complained. The clone
# is seconds; the generation it protects is hours.
if (CODE / '.git').is_dir():
    subprocess.run(['git', '-C', str(CODE), 'fetch', '--depth', '1', 'origin', 'main'],
                   capture_output=True, text=True)
    subprocess.run(['git', '-C', str(CODE), 'reset', '--hard', 'FETCH_HEAD'],
                   capture_output=True, text=True)
else:
    shutil.rmtree(CODE, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CODE)],
                   capture_output=True, text=True)

if not (CODE / 'model.py').exists():
    print('clone did not yield model.py; upload the framework .py files instead')
    from google.colab import files
    CODE.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (CODE / name).write_bytes(data)

sys.path.insert(0, str(CODE))

# Drop anything already imported, or a re-run of this cell keeps the stale module
# objects and the refresh above achieves nothing.
for _m in [m for m in list(sys.modules) if m.split('.')[0] in {
        'generator', 'physics', 'dataset', 'model', 'trainer', 'evaluate',
        'identifiability'}]:
    del sys.modules[_m]

head = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('code at', head or '(no git metadata)')

import dataset, model as model_mod, physics, trainer as trainer_mod, evaluate as eval_mod

# Fail here rather than mid-training. `aerodynamic_truth` is what section 9 scores
# against, and `fspb` must survive the split to reach it.
if not hasattr(physics, 'aerodynamic_truth'):
    raise RuntimeError(
        'stale physics.py: no aerodynamic_truth(). The clone did not refresh -- '
        'Runtime > Disconnect and delete runtime, then start again.'
    )
if 'fspb' not in dataset.TrajectoryDataset.__dataclass_fields__:
    raise RuntimeError('stale dataset.py: TrajectoryDataset has no fspb field.')

for m in (dataset, model_mod, physics, trainer_mod, eval_mod):
    print('ok:', pathlib.Path(m.__file__).name)


In [ ]:
REPO = 'https://github.com/alican30alicanexe-alt/systemid'
CODE = pathlib.Path('/content/systemid')

if not (CODE / 'model.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CODE)],
                   capture_output=True, text=True)

if not (CODE / 'model.py').exists():
    print('clone did not yield model.py; upload the framework .py files instead')
    from google.colab import files
    CODE.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (CODE / name).write_bytes(data)

sys.path.insert(0, str(CODE))

import dataset, model as model_mod, physics, trainer as trainer_mod, evaluate as eval_mod
for m in (dataset, model_mod, physics, trainer_mod, eval_mod):
    print('ok:', pathlib.Path(m.__file__).name)

## 3. Mount Drive and choose a dataset

Checkpoints go to Drive too — a 200-epoch run is long enough that a Colab
disconnect should not cost it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = pathlib.Path('/content/drive/MyDrive/systemid/data')
CKPT_DIR = pathlib.Path('/content/drive/MyDrive/systemid/checkpoints')
FIG_DIR  = pathlib.Path('/content/drive/MyDrive/systemid/figures')
for d in (CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

candidates = sorted(DATA_DIR.glob('*.npz'))
if not candidates:
    raise FileNotFoundError(
        f'no .npz under {DATA_DIR} -- run colab_generate_data.ipynb first')

for i, c in enumerate(candidates):
    print(f'  [{i}] {c.name:<28} {c.stat().st_size/1e6:8.1f} MB')

DATA = candidates[-1]   # newest by name; override by index if you want another
print('\nusing:', DATA)

## 4. The gate — does the residual grow with dynamic pressure?

This runs before the model is even built, because it decides whether training is
worth doing at all.

With `kinematics`, `gravity` and `propulsion` all analytical, the only physics left
unaccounted for is aerodynamic force. Aerodynamic force is proportional to dynamic
pressure. So the residual **must** be small in near-vacuum and rise through the
buckets. If it is flat, the residual is not aerodynamics — it is a bug in one of
the three "known" modules, and `dA` would absorb it silently.

Two other things worth watching in the output:

- **kinematic rows max abs error** should be ~1e-9 m/s or better. This is
  `d(SBII)/dt - VBII`, a pure finite-difference truncation check. Large values mean
  the generator's precision patch did not take.
- The bucket counts. If almost every sample sits in the lowest `pdynmc` bucket the
  comparison is weak regardless of what the medians say — most of a 190 s ascent is
  spent above the atmosphere.

In [ ]:
import numpy as np
from physics import residual_report, StateLayout, PhysicsModel, DEFAULT_KNOWN
from trainer import Q_BUCKETS

d = np.load(DATA, allow_pickle=True)
print('samples:', len(d['x']), ' runs:', len(np.unique(d['run_id'])),
      ' dtype:', d['x'].dtype)
print()

residual = residual_report(str(DATA))

# Re-derive the bucket medians so the gate is a value, not something read by eye.
layout = StateLayout(list(d['state_names']), list(d['param_names']))
accel  = residual[:, layout.s_slice('VBII')].norm(dim=1)
q      = torch.tensor(d['p'][:, layout.p('pdynmc')])

medians = []
for lo, hi in zip(Q_BUCKETS[:-1], Q_BUCKETS[1:]):
    sel = (q >= lo) & (q < hi)
    if sel.sum() > 10:
        medians.append((lo, hi, float(accel[sel].median()), int(sel.sum())))

print('\n' + '=' * 62)
rising = all(b[2] >= a[2] for a, b in zip(medians, medians[1:]))
span   = medians[-1][2] / max(medians[0][2], 1e-12) if len(medians) > 1 else float('nan')
print(f'monotonically rising in pdynmc : {rising}')
print(f'top bucket / bottom bucket     : {span:.1f}x')

GATE_OK = rising and len(medians) > 1 and span > 3.0
print(f'\nGATE_OK = {GATE_OK}')
if not GATE_OK:
    print('\nA flat or non-monotonic residual means an analytical module disagrees\n'
          'with CADAC. Training would learn that disagreement as aerodynamics.\n'
          'Check GravityJ2Module against cad_grav84 and confirm t reaches every\n'
          'module before continuing.')
print('=' * 62)

## 5. Split, and the shape of what gets learned

Splitting is **by trajectory, never by sample**. Consecutive samples are one plot
step apart and are near-duplicates; a per-sample split would place each validation
point beside a training point taken 10 ms earlier and report a flattering loss
however badly the model generalises to an unseen launch.

The mask printed below is the concrete answer to "what is frozen".

In [ ]:
from dataset import build_loaders
from model import GrayBoxSSM

BATCH_SIZE = 2048   # @param {type:"integer"}
SPLIT_SEED = 0      # @param {type:"integer"}

train_loader, val_loader, test, meta = build_loaders(
    DATA, batch_size=BATCH_SIZE, seed=SPLIT_SEED)

layout  = StateLayout(test.state_names, test.param_names)
known   = dict(DEFAULT_KNOWN)                 # aerodynamics is the only False
physics_model = PhysicsModel(layout, known=known)
print(physics_model)

mask = physics_model.free_mask()
print('\nfree_mask (1 = the network may write here)')
print('        ' + ' '.join(f'{n:>6}' for n in test.state_names))
for name, row in zip(test.state_names, mask.int().tolist()):
    print(f'{name:>7} ' + ' '.join(f'{v:>6}' for v in row))

print(f'\nfree dA entries : {int(mask.sum())} of {mask.numel()}')
print(f'free dc entries : {int((mask.sum(1) > 0).sum())} of {layout.n_state}')
print(f'equations/sample: 3   ->   {int(mask.sum()) + int((mask.sum(1) > 0).sum())}'
      ' unknowns against 3 equations')

## 6. Build the model

`fit_scalers` runs here, on **training batches only** — fitting on anything else
leaks the evaluation distribution into the normalisation.

It prints two vectors worth reading:

- `state RMS` — per-column scale of `x`. Position ~6.4e6, velocity ~1e3. This is why
  the network emits a dimensionless `A_tilde` and the code reconstructs
  `dA = diag(r) A_tilde diag(1/s)` afterwards. Emitting `dA` directly would need
  entries spanning six orders of magnitude and would not train.
- `residual RMS` — per-row scale of what analytical physics leaves behind. **Rows
  0–2 are finite-difference noise on frozen rows**, which is the cause of the MSE
  wart described in section 7.

The head is zero-initialised, so epoch 0 is *exactly* the analytical model.

In [ ]:
HIDDEN = [128, 128]   # @param
INIT_SEED = 0         # @param {type:"integer"}

torch.manual_seed(INIT_SEED)
gray = GrayBoxSSM.from_data(
    train_loader, physics_model, n_param=len(test.param_names), hidden=HIDDEN)

n_par = sum(p.numel() for p in gray.parameters())
print(f'\n[model] {n_par} parameters, head zero-initialised')

# Sanity: at init the gray-box must equal the analytical model exactly.
b = next(iter(val_loader))
with torch.no_grad():
    pred, parts = gray(b['x'], b['p'], b['t'])
delta = (pred - parts['xdot_known']).abs().max()
print(f'[model] |gray-box - analytical| at init = {delta:.3e}  (must be 0.0)')

## 7. Train

**Do not read the epoch MSE as progress.** `residual_scale`'s position rows are the
RMS of finite-difference noise and those rows are frozen, so they contribute ~1.0
each to the mean forever and the reported MSE sits near 0.5 regardless. Gradients
are unaffected — frozen rows have none — but the number is close to unreadable.

Judge from the two lines printed at the end instead:

```
[train] analytical : all=...  q0_10=...  q10_1000=...  q1000_10000=...  q10000_inf=...
[train] gray-box   : all=...  q0_10=...  q10_1000=...  q1000_10000=...  q10000_inf=...
```

Those are **median acceleration error in m/s², per dynamic-pressure regime**. The
one that matters is `q10000_inf` — max-Q, where aerodynamic force is largest and
therefore where there is actually something to learn. If `gray-box` does not beat
`analytical` there, nothing was identified no matter what the loss did.

`lambda_reg` is the identifiability knob, not a regularisation detail. At zero the
factorisation is non-unique and the recovered matrix is arbitrary among equally
good solutions. Section 9 sweeps it.

In [ ]:
# @title Training settings { run: "auto" }
EPOCHS      = 200     # @param {type:"integer"}
LR          = 2e-3    # @param {type:"number"}
LAMBDA_REG  = 1e-4    # @param {type:"number"}
PATIENCE    = 25      # @param {type:"integer"}
RUN_NAME    = 'graybox'  # @param {type:"string"}
IGNORE_GATE = False   # @param {type:"boolean"}

from trainer import TrainConfig, Trainer

if not GATE_OK and not IGNORE_GATE:
    raise RuntimeError(
        'residual gate did not pass -- see section 4. Training now would learn a '
        'physics-module error as aerodynamics. Set IGNORE_GATE=True to override '
        'deliberately.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = TrainConfig(epochs=EPOCHS, lr=LR, lambda_reg=LAMBDA_REG, patience=PATIENCE,
                  ckpt_dir=CKPT_DIR, run_name=RUN_NAME, device=device)
trainer = Trainer(gray, cfg, q_index=layout.p('pdynmc'))

t0 = time.monotonic()
history = trainer.fit(train_loader, val_loader)
print(f'\ntrained in {(time.monotonic() - t0) / 60:.1f} min on {device}')
print('checkpoint ->', trainer.ckpt_path)

## 8. Loss history

The absolute level is meaningless (section 7); the **shape** is not. A train/val gap
that opens means capacity or data limits; both curves flat from epoch 1 means the
network found nothing to add over analytical physics, which — if the gate passed —
points at `lambda_reg` being too large.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(history.train_mse, label='train')
axes[0].plot(history.val_mse, label='val')
axes[0].set_ylabel('normalised MSE  (~0.5 floor is the frozen-row artefact)')
axes[0].legend()

axes[1].semilogy(history.lr)
axes[1].set_ylabel('learning rate')

for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 9. What did it actually learn?

This is the section the project exists for. Everything above is machinery.

`delta_matrices` returns `dA` and `dc` in **physical units**, undoing the
conditioning sandwich exactly. The identified aerodynamic specific force is

```
a_ident = dA[3:6, 0:3] · r  +  dA[3:6, 3:6] · v  +  dc[3:6]      (m/s², inertial)
```

and CADAC's own value for the same quantity is in the dataset:

```
a_true  = ~TBI · (FSPB - FPB/vmass)          physics.aerodynamic_truth
```

so this is **measured, not inferred**. Three checks:

1. **Error against truth**, per dynamic-pressure bucket. The headline number. This
   is the answer to "how well did we identify the aerodynamics", with no modelling
   assumption anywhere in it.
2. **Direction agreement**, `cos(a_ident, a_true)` → should be **+1**. Note this
   replaces an earlier check that asked whether the force opposed velocity: true of
   drag, false of lift, so it penalised a correct model with a lift component.
   Against truth the assumption disappears.
3. **Where the force comes from.** `a_ident` splits exactly three ways —
   `dA[3:6,0:3]·r`, `dA[3:6,3:6]·v`, `dc[3:6]`. This is about *interpretability*,
   not accuracy: a model can match truth perfectly while putting the force in the
   position block, which is indistinguishable from a gravity-module error and makes
   the recovered matrix meaningless as a drag model. Drag belongs in the velocity
   block.

Checks 1–2 are correctness and can pass or fail outright. Check 3 is identifiability
and connects directly to section 11 — if it says `SUSPECT`, `lambda_reg` is the knob.

In [ ]:
from physics import aerodynamic_truth
from trainer import Q_BUCKETS

gray_cpu = gray.to('cpu')
N_PROBE = 20000   # @param {type:"integer"}

idx = torch.randperm(len(test.x))[:N_PROBE]
px, pp, pt, pf = test.x[idx], test.p[idx], test.t[idx], test.fspb[idx]
pq  = pp[:, layout.p('pdynmc')]
pos, vel = layout.s_slice('SBII'), layout.s_slice('VBII')

dA, dc = gray_cpu.delta_matrices(px, pp)
r, v = px[:, pos], px[:, vel]

# Exact decomposition of the identified aero force into its three sources.
a_pos = torch.bmm(dA[:, vel, pos], r.unsqueeze(-1)).squeeze(-1)
a_vel = torch.bmm(dA[:, vel, vel], v.unsqueeze(-1)).squeeze(-1)
a_off = dc[:, vel]
a_ident = a_pos + a_vel + a_off

a_true = aerodynamic_truth(pp, pf, layout, pt)          # CADAC's own value
err = (a_ident - a_true).norm(dim=1)
mi, mt = a_ident.norm(dim=1), a_true.norm(dim=1)
cos = (a_ident * a_true).sum(1) / (mi.clamp_min(1e-12) * mt.clamp_min(1e-12))
maxq = pq >= 1e4

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].loglog(mt.clamp_min(1e-6).numpy(), mi.clamp_min(1e-6).numpy(), '.', ms=1, alpha=0.3)
lims = [1e-4, max(float(mt.max()), float(mi.max())) * 1.5]
axes[0].plot(lims, lims, 'r--', lw=1, label='perfect')
axes[0].set_xlabel('|a_true| CADAC (m/s²)')
axes[0].set_ylabel('|a_ident| model (m/s²)')
axes[0].set_title('1. identified vs truth')
axes[0].legend()

axes[1].semilogx(pq.clamp_min(1e-3).numpy(), cos.numpy(), '.', ms=1, alpha=0.3)
axes[1].axhline(1, color='tab:red', ls='--', lw=0.8, label='perfect')
axes[1].set_xlabel('dynamic pressure (Pa)')
axes[1].set_ylabel('cos(a_ident, a_true)')
axes[1].set_ylim(-1.1, 1.1)
axes[1].set_title('2. direction agreement')
axes[1].legend()

if maxq.sum() > 10:
    block = dA[maxq][:, vel, vel].mean(0)
    lim = float(block.abs().max())
    im = axes[2].imshow(block.numpy(), cmap='RdBu_r', vmin=-lim, vmax=lim)
    axes[2].set_xticks(range(3), ['VBII1', 'VBII2', 'VBII3'])
    axes[2].set_yticks(range(3), ['VBII1', 'VBII2', 'VBII3'])
    axes[2].set_title('3. mean dA[3:6,3:6] at max-Q  (1/s)')
    fig.colorbar(im, ax=axes[2])

for ax in axes[:2]:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print('1. error against CADAC ground truth')
print(f"   {'pdynmc':<22}{'n':>8}{'|a_true|':>11}{'|error|':>11}{'rel':>9}")
for lo, hi in zip(Q_BUCKETS[:-1], Q_BUCKETS[1:]):
    s = (pq >= lo) & (pq < hi)
    if s.sum() > 10:
        tm, em = float(mt[s].median()), float(err[s].median())
        rel = f'{em / tm:8.1%}' if tm > 1e-6 else '       -'
        print(f'   {lo:>8.0f}-{hi:<9.0f} Pa {int(s.sum()):>8}{tm:>11.4f}{em:>11.4f}{rel}')

if maxq.sum() > 10:
    rel_maxq = float(err[maxq].median() / mt[maxq].median().clamp_min(1e-9))
    print(f'\n   {"OK" if rel_maxq < 0.25 else "POOR"} - {rel_maxq:.1%} median relative error at max-Q')

    med_cos = float(cos[maxq].median())
    print(f'\n2. direction')
    print(f'   median cos(a_ident, a_true) at max-Q  {med_cos:+.3f}')
    print(f'   {"OK - aligned with truth" if med_cos > 0.9 else "POOR - identified force points the wrong way"}')

    print(f'\n3. where the force comes from (median magnitude at max-Q, m/s²)')
    for label, part in (('dA[3:6,0:3] @ r  (gravity-shaped)', a_pos),
                        ('dA[3:6,3:6] @ v  (drag-shaped)   ', a_vel),
                        ('dc[3:6]          (offset)        ', a_off)):
        print(f'   {label}  {part[maxq].norm(dim=1).median():8.4f}')
    dominates = float(a_pos[maxq].norm(dim=1).median()) > float(a_vel[maxq].norm(dim=1).median())
    print('   ' + ('SUSPECT - position term dominates. The fit may still be accurate, but the\n'
                   '             matrix is not a drag model; raise lambda_reg (section 11).'
                   if dominates else 'OK - velocity term dominates, as drag should'))

## 10. Rollout against CADAC

One-step derivative error is not the quantity of interest — what matters is whether
the identified model reproduces a trajectory when integrated forward.

Every comparison carries an **integrator-matched floor**: the same forward-Euler
stepper driven by CADAC's own derivatives. CADAC integrates at `int_step` with a
second-order scheme while we roll out with Euler at `plot_step`, so a *perfect*
model still diverges purely from discretisation. A model sitting on the floor
cannot be improved by better identification — only by a better integrator.

⚠️ **This is slow.** `evaluate._euler` is a batch-size-1 Python loop: ~19,000 model
calls per rollout at `plot_step=0.01`, three rollouts per trajectory. Budget tens of
minutes *per trajectory*, and note that the GPU does not help. `N_EVAL_RUNS` limits
how many test trajectories are rolled out; start at 1.

In [ ]:
from dataset import TrajectoryDataset
from evaluate import evaluate

N_EVAL_RUNS = 1   # @param {type:"integer"}

def take_runs(ds, n):
    keep = torch.tensor(sorted(ds.run_id.unique().tolist())[:n])
    m = torch.isin(ds.run_id, keep)
    return TrajectoryDataset(x=ds.x[m], p=ds.p[m], t=ds.t[m], xdot=ds.xdot[m],
                             run_id=ds.run_id[m], fspb=ds.fspb[m],
                             state_names=ds.state_names, param_names=ds.param_names)

subset = take_runs(test, N_EVAL_RUNS)
print(f'rolling out {N_EVAL_RUNS} of {len(test.run_id.unique())} test runs, '
      f'{len(subset.x)} steps each\n')

t0 = time.monotonic()
table = evaluate(subset, gray_cpu, physics_model,
                 history_path=CKPT_DIR / f'{RUN_NAME}_history.json',
                 fig_dir=FIG_DIR)
print(f'\nrollout took {(time.monotonic() - t0) / 60:.1f} min')

# evaluate() writes with the Agg backend, so show the file rather than the figure.
from IPython.display import Image, display
display(Image(filename=str(FIG_DIR / 'evaluation.png')))

## 11. Identifiability — is the matrix meaningful, or just one of many?

Everything above can look good while the recovered matrix is arbitrary. With 21
unknowns and 3 equations per sample, infinitely many `dA` satisfy the data equally
well, and **the loss curve gives no warning at all**.

The test trains several seeds and compares them at identical probe points:

- **pred disagreement** — spread of `dA x + dc` across seeds. Should be small
  whenever the models fit. Measures agreement on the *dynamics*.
- **matrix disagreement** — spread of `A_tilde` across seeds. This is the actual
  identifiability question.

**Low prediction disagreement with high matrix disagreement is the failure
signature**: the seeds agree on the physics and disagree on the matrices, which
makes "interpretable" hollow. Raising `lambda_reg` selects the minimum-norm
correction and collapses the ambiguity; the sweep looks for the largest lambda that
buys that without costing `err maxQ`.

⚠️ Cost is `len(lambdas) × len(seeds)` full training runs. The defaults below are
4 × 3 = 12 runs at reduced epochs. Run it once you have a training result worth
trusting, not before.

In [ ]:
from identifiability import run_sweep, print_table

RUN_SWEEP    = False   # @param {type:"boolean"}
SWEEP_EPOCHS = 50      # @param {type:"integer"}
LAMBDAS      = [0.0, 1e-4, 1e-2, 1e-1]
SEEDS        = [0, 1, 2]

if RUN_SWEEP:
    t0 = time.monotonic()
    results = run_sweep(DATA, LAMBDAS, SEEDS, SWEEP_EPOCHS, CKPT_DIR / 'sweep')
    print_table(results, len(SEEDS))
    print(f'\nsweep took {(time.monotonic() - t0) / 60:.1f} min')
else:
    print('set RUN_SWEEP = True to run  '
          f'({len(LAMBDAS)} lambdas x {len(SEEDS)} seeds = '
          f'{len(LAMBDAS) * len(SEEDS)} training runs)')

## Next

Everything durable is on Drive: `checkpoints/<run_name>.pt`,
`checkpoints/<run_name>_history.json`, `figures/evaluation.png`.

To reload without retraining:

```python
trainer.load_checkpoint()          # same session
# or, fresh:
ckpt = torch.load(CKPT_DIR / 'graybox.pt', map_location='cpu', weights_only=False)
gray.load_state_dict(ckpt['model'])
```

**Ground truth is in the dataset — use it.** Section 9 infers that `dA x + dc` is
aerodynamic force *by elimination*. It no longer has to.

The `.npz` carries a `fspb` array: CADAC's own non-gravitational specific force in
body axes, deliberately kept out of `p` so the network cannot read the answer.
`newton.cpp` builds acceleration as `~TBI*FSPB + ~TGI*GRAVG`, so the true
aerodynamic specific force is

```python
truth = TBIᵀ @ (fspb - FPB/vmass)      # FPB = gimballed thrust, tvc.cpp 118-120
```

Comparing that against `dA x + dc` sample by sample replaces the three plausibility
checks with a measured error. The analytical modules already clear this test:
gravity plus propulsion reproduce CADAC to a median **4.0e-05 m/s²**, and **8e-06
m/s² at max-Q**, against a **5.25 m/s²** aerodynamic force there.